# Demo 1: Read the Matrix (SOLUTION)

**Module 01, Section 3. Instructor copy. Fully executed.**

Ten minutes, live. Load the Cordwell test set, build the confusion matrix,
read the four core metrics off it, then flip a single prediction by hand and
watch which numbers move and which do not.

Runs on CPU in under a second. No network, no model call, no Docker services.
That is by design: the fallback for this demo is this executed notebook.


## Setup

Imports and data load. Nothing to write here. The test set is 2,000 synthetic
Cordwell product reviews. Each row carries the review text, the ground truth
label (`y_true`: 1 means genuine safety issue, 0 means routine), and the
model's score (`y_score`: a number between 0 and 1 from a classifier that has
already been trained; higher means the model thinks safety issue).


In [ ]:
%pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

df = pd.read_csv("cordwell_test_set.csv")
y_test = df["y_true"].to_numpy()
y_score = df["y_score"].to_numpy()

print(f"{len(df)} reviews loaded")
print(f"{int(y_test.sum())} genuine safety escalations")
print(f"Base rate: {y_test.mean():.4f}")
df.head(3)


## Beat 1: A score is not a decision

The model hands us a score per review. Nobody can act on `0.7134`. The safety
team needs a yes or a no. Turning a score into a decision takes a threshold,
and for now we use the one `predict()` would silently pick for us: **0.50**.
Section 6 this afternoon interrogates that choice. Right now, just make it
explicit and visible instead of hidden inside a library default.


In [ ]:
THRESHOLD = 0.50

# A prediction is just a comparison: score at or above the cut means escalate.
y_pred = (y_score >= THRESHOLD).astype(int)

print(f"Reviews flagged for escalation: {y_pred.sum()} of {len(y_pred)}")


## Beat 2: The four numbers everything else is made of

Every metric this morning is arithmetic on four counts. `confusion_matrix`
returns them as a 2x2 grid; `.ravel()` flattens that grid into a flat list of
four numbers in a fixed order: **tn, fp, fn, tp**.

Memorize that order. Getting it backwards silently swaps precision and
recall, and the wrong number still looks plausible. We pass `labels=[0, 1]`
explicitly so the row and column order never depends on what happens to be in
the data.


In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

print(f"TP={tp}  FN={fn}  FP={fp}  TN={tn}")


Same four numbers as a picture. This cell is pre written; the plot is
boilerplate, the counts are the point. In Cordwell terms:

* **TP = 53**: safety issues flagged correctly. The system worked.
* **FN = 14**: safety issues missed. Fourteen reports sit unread. The costly cell.
* **FP = 107**: routine reviews flagged. Triage time wasted.
* **TN = 1,826**: routine reviews passed. Correct and cheap.


In [ ]:
if tp is not None:
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        labels=[0, 1],
        display_labels=["routine", "escalate"],
        colorbar=False,
    )


## Beat 3: Four metrics, read off the matrix

Each of these is a one line formula on the four counts. Compute them with
sklearn, then confirm one by hand so the room believes the formula and the
function are the same thing.

* Accuracy: the fraction of all 2,000 calls that were right.
* Precision: of everything we escalated, how much was genuinely a safety issue.
* Recall: of the 67 genuine safety issues, how many we caught.
* F1: the harmonic mean of precision and recall.


In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1       : {f1:.4f}")

# The formula and the function are the same thing:
print(f"Recall by hand: TP / (TP + FN) = {tp} / {tp + fn} = {tp / (tp + fn):.4f}")


The whole report in one call. `zero_division=0` matters the day a class gets
zero predictions: without it you get a warning and a silent `nan` in a
pipeline that keeps running. With it, 0 divided by 0 becomes 0.0.


In [ ]:
print(classification_report(
    y_test, y_pred,
    target_names=["routine", "escalate"],
    digits=3,
    zero_division=0,
))


## Beat 4: Flip one prediction

The memorable moment. Take exactly one of the 14 missed safety reports and
pretend the model had caught it. One review out of 2,000 changes its
prediction. Before running the next cell, ask the room: which metrics move,
and by how much?


In [ ]:
# Indices where the truth says safety issue but the model said routine.
fn_idx = np.where((y_test == 1) & (y_pred == 0))[0]
print(f"{len(fn_idx)} false negatives. Flipping the first one, row {fn_idx[0]}:")
print(f"  {df['review_text'].iloc[fn_idx[0]]!r}")

y_flipped = y_pred.copy()
y_flipped[fn_idx[0]] = 1


Recompute everything against the flipped predictions. The comparison table is
pre written. Watch the recall row and the accuracy row.


In [ ]:
def metric_row(name, fn, before, after):
    b, a = fn(y_test, before), fn(y_test, after)
    print(f"{name:<10} {b:.4f} -> {a:.4f}   ({(a - b) * 100:+.2f} points)")

if y_flipped is not None:
    print(f"{'metric':<10} {'before':>6}    {'after':>6}")
    metric_row("Accuracy", accuracy_score, y_pred, y_flipped)
    metric_row("Precision", precision_score, y_pred, y_flipped)
    metric_row("Recall", recall_score, y_pred, y_flipped)
    metric_row("F1", f1_score, y_pred, y_flipped)


One rescued safety report moved recall by one and a half points, because
recall's denominator is the 67 positives. The same rescued report moved
accuracy by five hundredths of a point, because accuracy's denominator is all
2,000 reviews and it is drowning in easy true negatives.

Accuracy did not notice the thing Cordwell cares about most. That is the
whole argument of the next section, delivered in one cell.


## Beat 5: The bridge. What does a model that never escalates score?

Slide 8 asked this before the break. A model that predicts routine for every
single review protects zero customers and reads zero safety reports. Score it.


In [ ]:
always_routine = np.zeros_like(y_test)

print(f"Accuracy of a model that never escalates: "
      f"{accuracy_score(y_test, always_routine):.4f}")
print(f"Accuracy of our actual model:             "
      f"{accuracy_score(y_test, y_pred):.4f}")


The do nothing model wins on accuracy by 2.7 points while missing all 67
safety reports. Accuracy is not wrong; it is answering a question nobody at
Cordwell asked. Section 4 names the metrics that survive this: balanced
accuracy and MCC. Leave that hanging; it is the next slide.


## Checks

Run this cell any time. It only reads variables you have already defined; it
never crashes on ones you have not.


In [ ]:
passed, total = 0, 0

def check(name, fn):
    global passed, total
    total += 1
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    mark = "PASS" if ok else "not yet"
    passed += ok
    print(f"[{mark:>7}] {name}")

check("data loaded (2000 rows, 67 positives)",
      lambda: len(df) == 2000 and int(y_test.sum()) == 67)
check("ST-1 y_pred is an int array of 0s and 1s",
      lambda: y_pred is not None and set(np.unique(y_pred)) <= {0, 1})
check("ST-1 y_pred flags 160 reviews at threshold 0.50",
      lambda: int(y_pred.sum()) == 160)
check("ST-2 ravel order is tn, fp, fn, tp",
      lambda: (tn, fp, fn, tp) == (1826, 107, 14, 53))
check("ST-3 accuracy is 0.9395",
      lambda: abs(acc - 0.9395) < 5e-5)
check("ST-3 precision is 0.3312",
      lambda: abs(prec - 0.33125) < 5e-5)
check("ST-3 recall is 0.7910",
      lambda: abs(rec - 53 / 67) < 5e-5)
check("ST-3 F1 is 0.4670",
      lambda: abs(f1 - 106 / 227) < 5e-5)
check("ST-5 found all 14 false negatives",
      lambda: fn_idx is not None and len(fn_idx) == 14)
check("ST-5 y_flipped rescues exactly one report",
      lambda: y_flipped is not None
              and int(y_flipped.sum()) == 161
              and int(y_pred.sum()) == 160)
check("ST-5 recall after the flip is 0.8060",
      lambda: abs(recall_score(y_test, y_flipped) - 54 / 67) < 5e-5)
check("ST-6 the do nothing baseline scores 0.9665",
      lambda: always_routine is not None
              and abs(accuracy_score(y_test, always_routine) - 0.9665) < 5e-5)

print(f"\n{passed} of {total} checks passing")


## Instructor appendix (optional, do not run live)

If a fast finisher asks what number they should trust instead of accuracy,
these two are the Section 4 answer. Previewing them here is fine one on one;
do not spend group time on them before the slides make the argument.


In [ ]:
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef

print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}")
print(f"MCC:               {matthews_corrcoef(y_test, y_pred):.4f}")
print(f"MCC of the do nothing baseline: "
      f"{matthews_corrcoef(y_test, np.zeros_like(y_test)):.4f}")
